# BANA 4373 — Lecture 4 Notebook  
## Data Preparation II: Merging (Joins) and Reshaping (Wide ↔ Long)

**Instructor:** Dr. Fidel González  
**Course:** BANA 4373 — Advanced Business Analytics for Economics and Business  
**Spring 2026**

### Goals for today
By the end of this notebook, you should be able to:
1. Define a **key** and verify whether it is unique.
2. Perform **left / inner / outer joins** in pandas and explain what changes.
3. Diagnose common merge problems (duplicates, type mismatches, missing matches).
4. Use `validate=` and `indicator=True` to catch merge issues early.
5. Reshape data between **wide** and **long** format using `pivot()` and `melt()`.
6. Save cleaned/merged outputs reproducibly using a project folder structure.

> **Important:** A merge can “work” but still be wrong. We will intentionally “break the merge” to learn how to detect problems.


## 0) Setup and project folders

We will create a **reproducible project structure** and keep all outputs inside it.

- `data_raw/`: original inputs (never edit)
- `data_clean/`: cleaned/merged outputs created by code
- `notebooks/`: this notebook and others
- `output/`: figures/tables/exports


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Display settings for readability (display only; does not change underlying numbers)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

project_root = Path("/Users/SHSU/Library/CloudStorage/Dropbox/Classes/ECON_4370/Lecture_4")
#project_root = Path("lecture4_project")
folders = ["data_raw", "data_clean", "notebooks", "output"]

for f in folders:
    (project_root / f).mkdir(parents=True, exist_ok=True)

print("✅ Project folders created:")
for p in sorted(project_root.iterdir()):
    print(" -", p)


## 1) Load a county dataset (base table)

For merging practice, we need a **base dataset** that represents our main sample.

We will try to:
1. Load a cleaned county file from `data_clean/` (if you saved one in Lecture 3), **or**
2. Pull Texas county data again from the Census API (quick fallback).

Our key will be:  
- `state` + `county` (together they uniquely identify a county)


In [ ]:
import requests

clean_path = project_root / "data_clean" / "acs_tx_counties_clean.csv"

if clean_path.exists():
    df_counties = pd.read_csv(clean_path)
    print(f"✅ Loaded cleaned county data from: {clean_path}")
else:
    print("ℹ️ Clean file not found. Pulling Texas counties from Census API as a fallback...")

    url = (
        "https://api.census.gov/data/2022/acs/acs5"
        "?get=NAME,B19013_001E,B01003_001E"
        "&for=county:*&in=state:48"
    )
    data = requests.get(url, timeout=30).json()
    df_counties = pd.DataFrame(data[1:], columns=data[0])

    # Convert to numeric
    for col in ["B19013_001E", "B01003_001E"]:
        df_counties[col] = pd.to_numeric(df_counties[col], errors="coerce")

    # Validity flag (income must be > 0 to be plausible)
    df_counties["income_valid"] = df_counties["B19013_001E"] > 0
    df_counties.loc[~df_counties["income_valid"], "B19013_001E"] = pd.NA

    # Rename to readable names
    df_counties = df_counties.rename(columns={
        "B19013_001E": "median_household_income",
        "B01003_001E": "population"
    })

    print("✅ Pulled and cleaned Census county data.")

df_counties.head()


## 2) Quick quality checks (before any merge)

Before merging, always check:
- Shape
- Duplicates
- Missing values
- Key uniqueness


In [ ]:
print("Shape (rows, cols):", df_counties.shape)
print("Duplicate FULL rows:", df_counties.duplicated().sum())
print("\nMissing values by column:")
display(df_counties.isna().sum())

# Key uniqueness check: state+county should uniquely identify each row
key_dups = df_counties.duplicated(subset=["state", "county"]).sum()
print("\nDuplicate KEY rows (state+county):", key_dups)

# Show a few key columns
display(df_counties[["NAME", "state", "county", "median_household_income", "population"]].head(5))


## 3) Save the base dataset (reproducibly)

We will save the cleaned county table so merges are reproducible without re-pulling from the API each time.


In [ ]:
base_out = project_root / "data_clean" / "counties_base.csv"
df_counties.to_csv(base_out, index=False)
print(f"✅ Saved base counties file to: {base_out}")


## 4) Merging: left / inner / outer joins (concept + practice)

We will create a second dataset that we want to merge into counties.

### Example: A small county-level dataset
Suppose we have another file with a county-level variable (e.g., a fabricated "property_tax_rate").

We will practice:
- Left join (most common in applied work)
- Inner join (drops non-matching rows)
- Outer join (useful for diagnostics)


In [ ]:
# Create a SECOND dataset with a subset of counties (intentionally not all)
np.random.seed(42)

df_tax = df_counties[["state", "county"]].sample(200, random_state=42).copy()
df_tax["property_tax_rate"] = np.round(np.random.uniform(0.8, 2.8, size=len(df_tax)), 3)

print("Second dataset shape:", df_tax.shape)
df_tax.head()


In [ ]:
# LEFT JOIN: keep all counties, add tax info where available
left_join = df_counties.merge(df_tax, on=["state", "county"], how="left")
print("Left join shape:", left_join.shape)
print("Missing property_tax_rate after left join:", left_join["property_tax_rate"].isna().sum())

# INNER JOIN: keep only counties that appear in BOTH
inner_join = df_counties.merge(df_tax, on=["state", "county"], how="inner")
print("\nInner join shape:", inner_join.shape)

# OUTER JOIN: keep all rows from both sides (diagnostic)
outer_join = df_counties.merge(df_tax, on=["state", "county"], how="outer", indicator=True)
print("\nOuter join shape:", outer_join.shape)
outer_join["_merge"].value_counts()


### Interpreting `_merge` from `indicator=True`

- `both`: matched keys in both datasets  
- `left_only`: in counties but not in tax dataset  
- `right_only`: in tax dataset but not in counties (should not happen here if tax was built from counties)


## 5) Break the merge examples (common real-world failures)

We will intentionally create mistakes that cause merges to fail or silently produce incorrect results.

### We will demonstrate three classic problems:
1. **Duplicate keys** on the right (many-to-many merge risk)
2. **Mismatched types** (string vs numeric IDs)
3. **Wrong key choice** (merging on NAME instead of FIPS)


In [ ]:
# -------------------------
# (1) Duplicate keys example
# -------------------------

# Create duplicates in the right dataset by repeating some rows
df_tax_dup = pd.concat([df_tax, df_tax.sample(20, random_state=1)], ignore_index=True)

print("df_tax original shape:", df_tax.shape)
print("df_tax_dup shape (with duplicates):", df_tax_dup.shape)

# Check duplicate keys
dup_count = df_tax_dup.duplicated(subset=["state", "county"]).sum()
print("Duplicate keys in df_tax_dup:", dup_count)

# Attempt a merge with validate=... to catch the problem
try:
    test = df_counties.merge(df_tax_dup, on=["state", "county"], how="left", validate="one_to_one")
except Exception as e:
    print("✅ Merge failed as expected with validate='one_to_one'")
    print("Error message:", e)


In [ ]:
# Fix duplicates by keeping the first occurrence (simple fix for lecture demo)
df_tax_nodup = df_tax_dup.drop_duplicates(subset=["state", "county"], keep="first").copy()

# Now validate succeeds
merged_ok = df_counties.merge(df_tax_nodup, on=["state", "county"], how="left", validate="one_to_one")
print("✅ Merge succeeded after removing duplicate keys.")
print("Merged shape:", merged_ok.shape)


In [ ]:
# -------------------------
# (2) Mismatched types example
# -------------------------

# Make a copy and break the types: convert county to int in the right dataset
df_tax_badtype = df_tax.copy()
df_tax_badtype["county"] = df_tax_badtype["county"].astype(int)   # int

print("df_counties county dtype:", df_counties["county"].dtype)
print("df_tax_badtype county dtype:", df_tax_badtype["county"].dtype)

# Merge (this will create many missing matches because '001' != 1, and string != int)
bad_merge = df_counties.merge(df_tax_badtype, on=["state", "county"], how="left")
print("\nMissing property_tax_rate due to type mismatch:", bad_merge["property_tax_rate"].isna().sum())
print("Total rows:", len(bad_merge))

# Diagnostic: show a few rows that did not match
bad_merge[bad_merge["property_tax_rate"].isna()][["NAME", "state", "county"]].head()


In [ ]:
# Fix: force both sides to the same type (strings are safest for codes)
df_tax_fix = df_tax_badtype.copy()
df_tax_fix["county"] = df_tax_fix["county"].astype(str).str.zfill(3)  # ensure 3 digits
df_tax_fix["state"] = df_tax_fix["state"].astype(str).str.zfill(2)

df_counties_fix = df_counties.copy()
df_counties_fix["county"] = df_counties_fix["county"].astype(str).str.zfill(3)
df_counties_fix["state"] = df_counties_fix["state"].astype(str).str.zfill(2)

good_merge = df_counties_fix.merge(df_tax_fix, on=["state", "county"], how="left", validate="one_to_one")
print("✅ Type mismatch fixed.")
print("Missing property_tax_rate now:", good_merge["property_tax_rate"].isna().sum())


In [ ]:
# -------------------------
# (3) Wrong key choice example (using NAME)
# -------------------------

# Create a 'fake' second dataset that has NAME with slightly different formatting
df_namekey = df_counties[["NAME"]].sample(200, random_state=7).copy()
df_namekey["some_value"] = np.random.randint(1, 100, size=len(df_namekey))

# Introduce formatting differences to break name-based merge
df_namekey["NAME"] = df_namekey["NAME"].str.replace(", Texas", " County, TX", regex=False)

# Merge on NAME (bad idea) -> many missing matches
bad_name_merge = df_counties.merge(df_namekey, on="NAME", how="left")

print("Rows:", len(bad_name_merge))
print("Missing some_value after NAME merge:", bad_name_merge["some_value"].isna().sum())
bad_name_merge[["NAME", "some_value"]].head(10)


**Lesson:** Names are not stable identifiers.  
Even small differences in punctuation or abbreviations can destroy a merge.

**Best practice:** Use codes (FIPS) and keep them as strings with leading zeros.


## 6) Reshaping data: wide ↔ long

Economics datasets often arrive in wide format (one column per year) but analysis usually needs long format.

We will create a small **toy panel** for a few counties and show:
- `pivot()` (long → wide)
- `melt()` (wide → long)


In [ ]:
# Pick a few counties for a toy example
sample_keys = df_counties_fix[["state", "county", "NAME"]].sample(5, random_state=0).copy()

years = [2019, 2020, 2021, 2022]
rows = []
np.random.seed(0)

for _, r in sample_keys.iterrows():
    base = np.random.uniform(3.0, 7.0)  # baseline unemployment rate
    for y in years:
        shock = np.random.normal(0, 0.6)
        rows.append({
            "state": r["state"],
            "county": r["county"],
            "NAME": r["NAME"],
            "year": y,
            "unemp_rate": round(max(0.0, base + shock), 2)
        })

df_unemp_long = pd.DataFrame(rows)
print("Long panel shape:", df_unemp_long.shape)
df_unemp_long.head(8)


In [ ]:
# LONG -> WIDE using pivot (one row per county, columns are years)
df_unemp_wide = df_unemp_long.pivot(index=["state", "county", "NAME"], columns="year", values="unemp_rate").reset_index()
df_unemp_wide.columns.name = None  # remove the 'year' label from columns

print("Wide shape:", df_unemp_wide.shape)
df_unemp_wide.head()


In [ ]:
# WIDE -> LONG using melt
df_back_to_long = df_unemp_wide.melt(
    id_vars=["state", "county", "NAME"],
    value_vars=years,
    var_name="year",
    value_name="unemp_rate"
).sort_values(["state", "county", "year"])

print("Back-to-long shape:", df_back_to_long.shape)
df_back_to_long.head(10)


### Quick check: did we recover the same long data?

We should get the same rows back (possibly in a different order).


In [ ]:
a = df_unemp_long.sort_values(["state", "county", "year"]).reset_index(drop=True)
b = df_back_to_long.sort_values(["state", "county", "year"]).reset_index(drop=True)

print("Same shape?", a.shape == b.shape)
print("All equal?", a.equals(b))
print("✅ Reshape check complete")


## 7) Build an analysis-ready panel (mini workflow)

Now we combine:
- County base data (income, population)
- Toy unemployment panel (long format)

This simulates what we will do in future weeks with real BLS LAUS panels.


In [ ]:
panel = df_unemp_long.merge(
    df_counties_fix[["state", "county", "median_household_income", "population"]],
    on=["state", "county"],
    how="left",
    validate="many_to_one"  # many rows per county in panel, one row per county in base
)

print("Panel shape:", panel.shape)
print("Missing income after merge:", panel["median_household_income"].isna().sum())

panel.head()


In [ ]:
panel_out = project_root / "data_clean" / "toy_county_panel.csv"
panel.to_csv(panel_out, index=False)
print(f"✅ Saved toy panel to: {panel_out}")
print("✅ Notebook finished running")


## Key takeaways

- Always define your key and check uniqueness *before* merging.
- Prefer coded identifiers (FIPS) over names.
- Use `validate=` to catch bad merges early.
- Use `indicator=True` to diagnose unmatched rows.
- Reshaping (wide ↔ long) is essential for panel analysis and plotting.
